In [ ]:
import torch
import numpy as np
import pandas as pd
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn as nn

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))])

In [ ]:
trainDataset = datasets.MNIST(root='data', download=True, transform=transform, train=True)
testDataset = datasets.MNIST(root='data', download=True, transform=transform, train=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 64.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.79MB/s]


In [ ]:
trainDataLoader = DataLoader(trainDataset, batch_size=64, shuffle=True)

In [ ]:
testDataLoader = DataLoader(testDataset, batch_size=64, shuffle=True)

In [ ]:
class ConvNet(nn.Module):
  def __init__(self, inputSize):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Flatten(),
        nn.Linear(in_features=1152, out_features=128),
        nn.ReLU(),
        nn.Linear(in_features=128, out_features=10)
    )

  def forward(self,x):
    return self.Model(x)


In [ ]:
convNet = ConvNet(inputSize=1).to(device)

In [ ]:
criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(convNet.parameters(), lr=0.0001)

In [ ]:
epochs = 10

In [ ]:
for epoch in range(epochs):
  totalLoss = 0.0
  batchSize = 0.0
  for img, label in trainDataLoader:

    img = img.to(device)
    label = label.to(device)

    op = convNet(img)
    loss = criteria(op, label)

    totalLoss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  batchSize = len(trainDataLoader)
  avgLoss = totalLoss/batchSize
  print(f"epoch: {epoch+1}/{epochs}, Loss: {avgLoss:.4}")


epoch: 1/10, Loss: 0.4848
epoch: 2/10, Loss: 0.1091
epoch: 3/10, Loss: 0.07488
epoch: 4/10, Loss: 0.05856
epoch: 5/10, Loss: 0.04804
epoch: 6/10, Loss: 0.04092
epoch: 7/10, Loss: 0.03552
epoch: 8/10, Loss: 0.03209
epoch: 9/10, Loss: 0.02748
epoch: 10/10, Loss: 0.02507


In [ ]:
convNet.eval()
correct = 0
total = 0

with torch.no_grad():
  for img, label in testDataLoader:
    img = img.to(device)
    label = label.to(device)

    op = convNet(img)
    _,prediction = torch.max(op,1)

    correct += (prediction == label).sum().item()
    total += label.size(0)

print(f"Accuracy: {(correct/total):.2%}")


Accuracy: 98.71%
